# 02 — Fetch Players & Games

Loops over all 32 teams: pulls each roster into `players`, and each season schedule into `games`.

In [5]:
from google.colab import drive
drive.mount('/content/drive')

import sys
PROJECT_DIR = '/content/drive/MyDrive/NHL_Analytics_Hub'
sys.path.append(PROJECT_DIR)
import nhl_api_utils as nhl

DB_PATH = f'{PROJECT_DIR}/data/nhl_hockey.db'
conn = nhl.get_connection(DB_PATH)


Mounted at /content/drive


## Step 1 — Inspect one roster response

In [6]:
sample = nhl.fetch_roster('COL')
print(list(sample.keys()))
sample['forwards'][0]


['forwards', 'defensemen', 'goalies']


{'id': 8481550,
 'headshot': 'https://assets.nhle.com/mugs/nhl/20262027/COL/8481550.png',
 'firstName': {'default': 'Adam'},
 'lastName': {'default': 'Beckman'},
 'positionCode': 'L',
 'shootsCatches': 'L',
 'heightInInches': 74,
 'weightInPounds': 192,
 'heightInCentimeters': 188,
 'weightInKilograms': 87,
 'birthDate': '2001-05-10',
 'birthCity': {'default': 'Saskatoon'},
 'birthCountry': 'CAN',
 'birthStateProvince': {'default': 'SK'}}

In [7]:
sample_schedule = nhl.fetch_schedule('COL')

print(list(sample_schedule.keys()))

sample_schedule['games'][0]

['previousSeason', 'currentSeason', 'nextSeason', 'clubTimezone', 'clubUTCOffset', 'games']


{'id': 2025010103,
 'season': 20252026,
 'gameType': 1,
 'gameDate': '2025-09-21',
 'venue': {'default': 'Magness Arena'},
 'neutralSite': True,
 'startTimeUTC': '2025-09-21T23:00:00Z',
 'easternUTCOffset': '-04:00',
 'venueUTCOffset': '-06:00',
 'venueTimezone': 'America/Denver',
 'gameState': 'FINAL',
 'gameScheduleState': 'OK',
 'tvBroadcasts': [],
 'awayTeam': {'id': 21,
  'commonName': {'default': 'Avalanche'},
  'placeName': {'default': 'Colorado'},
  'placeNameWithPreposition': {'default': 'Colorado', 'fr': 'du Colorado'},
  'abbrev': 'COL',
  'logo': 'https://assets.nhle.com/logos/nhl/svg/COL_light.svg',
  'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/COL_dark.svg',
  'awaySplitSquad': True,
  'score': 5},
 'homeTeam': {'id': 68,
  'commonName': {'default': 'Mammoth'},
  'placeName': {'default': 'Utah'},
  'placeNameWithPreposition': {'default': 'Utah', 'fr': "de l'Utah"},
  'abbrev': 'UTA',
  'logo': 'https://assets.nhle.com/logos/nhl/svg/UTA_light.svg?season=20252026',
 

## Step 2 — Loop all teams, flatten forwards/defensemen/goalies into `players`

In [8]:
players_loaded = 0

for abbrev in nhl.TEAM_ABBREVS:
    team_id = nhl.get_team_id(conn, abbrev)
    if team_id is None:
        print(f'  skipping {abbrev} — not found in teams table (run notebook 01 first)')
        continue

    roster = nhl.fetch_roster(abbrev)
    if not roster:
        continue

    all_players = roster.get('forwards', []) + roster.get('defensemen', []) + roster.get('goalies', [])

    for p in all_players:
        row = {
            'player_id': p['id'],
            'team_id': team_id,
            'first_name': p['firstName']['default'],
            'last_name': p['lastName']['default'],
            'position': p.get('positionCode'),
            'jersey_number': p.get('sweaterNumber'),
            'birth_date': p.get('birthDate'),
            'birth_country': p.get('birthCountry'),
            'height_cm': p.get('heightInCentimeters'),
            'weight_kg': p.get('weightInKilograms'),
            'shoots_catches': p.get('shootsCatches'),
            'headshot_url': p.get('headshot'),
        }
        nhl.insert_ignore(conn, 'players', row)
        players_loaded += 1

    conn.commit()
    print(f'{abbrev}: {len(all_players)} players')

print(f'TOTAL players processed: {players_loaded}')


ANA: 46 players
BOS: 43 players
BUF: 44 players
CGY: 40 players
CAR: 42 players
CHI: 44 players
COL: 42 players
CBJ: 44 players
DAL: 43 players
DET: 44 players
EDM: 45 players
FLA: 44 players
LAK: 53 players
MIN: 45 players
MTL: 42 players
NSH: 44 players
NJD: 44 players
NYI: 44 players
NYR: 27 players
OTT: 27 players
PHI: 44 players
PIT: 23 players
SJS: 42 players
SEA: 26 players
STL: 47 players
TBL: 49 players
TOR: 47 players
UTA: 26 players
VAN: 45 players
VGK: 34 players
WSH: 25 players
WPG: 43 players
TOTAL players processed: 1298


In [9]:
# test
import pandas as pd
print(pd.read_sql(
    'SELECT COUNT(*) AS n FROM players',
    conn
))

      n
0  1298


In [10]:
# test
print(pd.read_sql("""
SELECT player_id, COUNT(*) AS occurrences
FROM players
GROUP BY player_id
HAVING COUNT(*) > 1
""", conn))

Empty DataFrame
Columns: [player_id, occurrences]
Index: []


In [11]:
# test
print(pd.read_sql("""
SELECT COUNT(*) AS n
FROM players p
LEFT JOIN teams t
    ON p.team_id = t.team_id
WHERE t.team_id IS NULL
""", conn))

   n
0  0


In [12]:
# test
print(pd.read_sql("""
SELECT
    t.team_abbrev,
    t.team_name,
    COUNT(p.player_id) AS player_count
FROM teams t
LEFT JOIN players p
    ON t.team_id = p.team_id
GROUP BY t.team_id, t.team_abbrev, t.team_name
ORDER BY t.team_abbrev;
""", conn))

   team_abbrev              team_name  player_count
0          ANA          Anaheim Ducks            46
1          BOS          Boston Bruins            43
2          BUF         Buffalo Sabres            44
3          CAR    Carolina Hurricanes            42
4          CBJ  Columbus Blue Jackets            44
5          CGY         Calgary Flames            40
6          CHI     Chicago Blackhawks            44
7          COL     Colorado Avalanche            42
8          DAL           Dallas Stars            43
9          DET      Detroit Red Wings            44
10         EDM        Edmonton Oilers            45
11         FLA       Florida Panthers            44
12         LAK      Los Angeles Kings            53
13         MIN         Minnesota Wild            45
14         MTL     Montréal Canadiens            42
15         NJD      New Jersey Devils            44
16         NSH    Nashville Predators            44
17         NYI     New York Islanders            44
18         N

## Step 3 — Loop all teams, load `games` from each club's season schedule

Each game appears in two teams' schedules — `INSERT OR IGNORE` (via `game_id` PK) safely dedupes it.

In [13]:
games_loaded = 0

for abbrev in nhl.TEAM_ABBREVS:
    schedule = nhl.fetch_schedule(abbrev)
    if not schedule:
        continue

    for g in schedule.get('games', []):
        home_abbrev = g['homeTeam']['abbrev']
        away_abbrev = g['awayTeam']['abbrev']
        home_id = nhl.get_team_id(conn, home_abbrev)
        away_id = nhl.get_team_id(conn, away_abbrev)

        row = {
            'game_id': g['id'],
            'season': str(g.get('season')),
            'game_type': g.get('gameType'),
            'game_date': g.get('gameDate'),
            'home_team_id': home_id,
            'away_team_id': away_id,
            'home_score': g['homeTeam'].get('score'),
            'away_score': g['awayTeam'].get('score'),
            'game_state': g.get('gameState'),
            'venue_name': g.get('venue', {}).get('default') if isinstance(g.get('venue'), dict) else g.get('venue'),
        }
        nhl.insert_ignore(conn, 'games', row)
        games_loaded += 1

    conn.commit()
    print(f'{abbrev} schedule processed')

print(f'TOTAL game rows processed (pre-dedupe count): {games_loaded}')


ANA schedule processed
BOS schedule processed
BUF schedule processed
CGY schedule processed
CAR schedule processed
CHI schedule processed
COL schedule processed
CBJ schedule processed
DAL schedule processed
DET schedule processed
EDM schedule processed
FLA schedule processed
LAK schedule processed
MIN schedule processed
MTL schedule processed
NSH schedule processed
NJD schedule processed
NYI schedule processed
NYR schedule processed
OTT schedule processed
PHI schedule processed
PIT schedule processed
SJS schedule processed
SEA schedule processed
STL schedule processed
TBL schedule processed
TOR schedule processed
UTA schedule processed
VAN schedule processed
VGK schedule processed
WSH schedule processed
WPG schedule processed
TOTAL game rows processed (pre-dedupe count): 2996


In [15]:
import pandas as pd
print(pd.read_sql('SELECT COUNT(*) AS n FROM games', conn))

      n
0  1498


In [19]:
#test
print(pd.read_sql('SELECT game_id, COUNT(*) FROM games GROUP BY game_id HAVING COUNT(*) > 1', conn))
# SELECT game_id, COUNT(*)
# FROM games
# GROUP BY game_id
# HAVING COUNT(*) > 1;

Empty DataFrame
Columns: [game_id, COUNT(*)]
Index: []


In [20]:
#test

SELECT COUNT(*)
FROM games g
LEFT JOIN teams t
    ON g.home_team_id = t.team_id
WHERE t.team_id IS NULL;

SyntaxError: Invalid star expression (238565043.py, line 2)

In [21]:
print(pd.read_sql("""
SELECT COUNT(*)
FROM games g
LEFT JOIN teams t
    ON g.home_team_id = t.team_id
WHERE t.team_id IS NULL
""", conn))

   COUNT(*)
0         0


In [22]:
print(pd.read_sql("""
SELECT COUNT(*)
FROM games g
LEFT JOIN teams t
    ON g.away_team_id = t.team_id
WHERE t.team_id IS NULL
""", conn))

   COUNT(*)
0         0


In [23]:
import pandas as pd
print(pd.read_sql('SELECT COUNT(*) AS n FROM players', conn))
print(pd.read_sql('SELECT COUNT(*) AS n FROM games', conn))
pd.read_sql('SELECT * FROM games LIMIT 5', conn)


      n
0  1298
      n
0  1498


,game_id,season,game_type,game_date,home_team_id,away_team_id,home_score,away_score,game_state,venue_name
0,2025010001,20252026,1,2025-09-21,20,18,3,1,FINAL,Toyota Arena
1,2025010002,20252026,1,2025-09-21,14,29,2,3,FINAL,Rogers Place
2,2025010003,20252026,1,2025-09-21,29,14,0,3,FINAL,Scotiabank Saddledome
3,2025010004,20252026,1,2025-09-21,23,25,5,0,FINAL,Bridgestone Arena
4,2025010005,20252026,1,2025-09-21,23,25,5,3,FINAL,Bridgestone Arena


In [ ]:
conn.close()
